# MinneApple — Entrenamiento de respaldo en Colab (1280 px, GPU T4)
Proyecto 2 de Visión Computacional con Deep Learning (UAO)

**Para qué sirve este notebook.** El entrenamiento principal corre en un portátil con una GTX 1050 (4 GB), que no alcanza para entrenar a
1280 px (la resolución nativa del dataset). Colab da una GPU con más memoria (T4, 16 GB o mejor), así que aquí se prueba **un tiro extra**:
el mismo modelo (nano, con self-attention C2PSA), a **imgsz=1280** en vez de 1024, con los mismos hiperparámetros que las decisiones 0005 y 0016.

**No depende de nuestro repositorio git** (todavía no tiene remoto): esta celda a celda descarga los mismos datos de la Universidad de
Minnesota y reconstruye **la misma partición train/val por video** que se usa en local (decisión 0004), copiada literal, para que el
resultado sea comparable.

**Cómo usarlo:**
1. Sube este archivo a Google Colab (colab.research.google.com → Subir cuaderno) o ábrelo desde Drive.
2. Entorno de ejecución → Cambiar tipo de entorno → **GPU** (T4 o mejor).
3. Ejecuta las celdas en orden, de arriba hacia abajo.
4. Al final, descarga `best.pt`, `results.csv` y `args.yaml` (instrucciones en la última celda) y colócalos en
   `runs/colab_1280_s0/weights/` (los dos primeros) y `runs/colab_1280_s0/` (el tercero) de tu copia local del repositorio.
5. Evalúa con `python scripts/evaluar.py --nombre colab_1280_s0 --particion val --conf 0.35` y compáralo con la regla de la decisión 0016.

## 0. Preparación del entorno

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No hay GPU. Entorno de ejecución -> Cambiar tipo de entorno -> GPU (T4 o mejor), y vuelve a ejecutar esta celda."
    )
nombre_gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {nombre_gpu} | VRAM: {vram_gb:.1f} GB")
if vram_gb < 8:
    print("Aviso: menos de 8 GB de VRAM. Puede que haya que bajar --batch más adelante (celda de entrenamiento).")

### 0.1 Conectar Google Drive
Los resultados (pesos, métricas) se guardan directo en Drive: si Colab se desconecta a mitad del entrenamiento, el progreso no se pierde
(Ultralytics puede reanudar desde el último checkpoint).

In [ ]:
from pathlib import Path

RAIZ_DATOS = Path("/content/data")  # dataset: disco local de la sesión (más rápido para leer miles de veces)
RAIZ_DRIVE = Path("/content/drive/MyDrive/minneapple_colab")  # pesos y métricas: persisten aunque se desconecte

from google.colab import drive
drive.mount("/content/drive")
RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)
print("Drive listo en", RAIZ_DRIVE)

### 0.2 Instalar Ultralytics (misma versión que el entorno local)

In [ ]:
!pip install --quiet "ultralytics==8.4.149" opencv-python-headless pillow numpy

## 1. Descargar MinneApple
Mismos archivos, mismos MD5 y misma licencia (CC BY-NC-SA 3.0 US) que en el README del repositorio local. Solo `detection.tar.gz`:
`test_data.zip` no hace falta para entrenar (train/val salen de `detection/train`).

In [ ]:
RAIZ_DATOS.mkdir(parents=True, exist_ok=True)
detection = RAIZ_DATOS / "detection.tar.gz"
if not detection.exists():
    !curl -L -A "Mozilla/5.0" -o {detection} "https://conservancy.umn.edu/server/api/core/bitstreams/3ef26f04-6467-469b-9857-f443ffa1bb61/content"

import hashlib
md5 = hashlib.md5(detection.read_bytes()).hexdigest()
assert md5 == "d24a2f70144f1a3c5a52ffef66c50630", f"MD5 no coincide: {md5} (¿descarga incompleta o corrupta?)"
print("MD5 verificado:", md5)

if not (RAIZ_DATOS / "detection" / "train" / "images").exists():
    !tar -xzf {detection} -C {RAIZ_DATOS}
print("Imágenes de train:", len(list((RAIZ_DATOS / "detection" / "train" / "images").glob("*.png"))))

## 2. Partición y conversión a YOLO-seg
**Copiado literal** de `scripts/preparar_datos.py` (funciones `group_of`, `mask_to_polygons` y el conjunto `VAL_GROUPS`), para que la
partición train/val sea exactamente la misma que en local (decisión 0004) y el resultado sea comparable.

In [ ]:
import os
import shutil

import cv2
import numpy as np
from PIL import Image
from ultralytics.data.converter import merge_multi_segment

OUT = Path("/content/data/yolo")
# Idéntico a scripts/preparar_datos.py (decisión 0004): dos videos completos para validación.
VAL_GROUPS = {"20150921_131234", "20150921_131729"}


def group_of(name: str) -> str:
    return name.rsplit("_", 1)[0]


def mask_to_polygons(mask: np.ndarray):
    h, w = mask.shape
    polys = []
    for inst in np.unique(mask)[1:]:
        m = (mask == inst).astype(np.uint8)
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = [c.reshape(-1, 2) for c in cnts if len(c) >= 3]
        if not cnts:
            continue
        seg = np.concatenate(merge_multi_segment(cnts)) if len(cnts) > 1 else cnts[0]
        polys.append(seg / [w, h])
    return polys


if OUT.exists():
    shutil.rmtree(OUT)
img_dir = RAIZ_DATOS / "detection" / "train" / "images"
mask_dir = RAIZ_DATOS / "detection" / "train" / "masks"
grupos_por_particion = {"train": set(), "val": set()}
for img_path in sorted(img_dir.glob("*.png")):
    g = group_of(img_path.name)
    split = "val" if g in VAL_GROUPS else "train"
    grupos_por_particion[split].add(g)
    mask = np.array(Image.open(mask_dir / img_path.name))
    polys = mask_to_polygons(mask)
    for sub in ("images", "labels"):
        (OUT / sub / split).mkdir(parents=True, exist_ok=True)
    os.link(img_path, OUT / "images" / split / img_path.name)
    lineas = ["0 " + " ".join(f"{v:.6f}" for v in p.reshape(-1)) for p in polys]
    (OUT / "labels" / split / f"{img_path.stem}.txt").write_text("\n".join(lineas))

# Mismo assert que en local: ningún video en ambas particiones.
assert not (grupos_por_particion["train"] & grupos_por_particion["val"]), "un video quedó en las dos particiones"
n_train = len(list((OUT / "images" / "train").glob("*.png")))
n_val = len(list((OUT / "images" / "val").glob("*.png")))
assert n_train == 577 and n_val == 93, f"conteo distinto al local: train={n_train} val={n_val} (esperado 577/93)"
(OUT / "minneapple.yaml").write_text(f"path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n  0: apple\n")
print(f"OK: train={n_train}, val={n_val} (coincide con la partición local, decisión 0004)")

## 3. Entrenamiento a 1280 px
Mismos hiperparámetros que la base local (decisiones 0005 y 0016): `mask_ratio=8`, sin parada temprana, determinista, semilla 0,
`scale=0.5`. Cambia `imgsz` (1280 en vez de 1024), el lote (más alto porque hay más VRAM) y, opcionalmente, la **escala del modelo**
(`nano` o `small`). **Aviso:** correr `small` aquí combina DOS cambios a la vez (tamaño de modelo y resolución) frente a la base local; se compara con la base (nano-1024) y con `colab_1280_s0` (nano-1280, ya evaluado y sin mejora suficiente: +0,0017 de caja, no cumple la decisión 0016) para poder separar qué aporta cada uno. Los resultados se escriben
directo en Drive.

In [ ]:
from ultralytics import YOLO

ESCALA = "s"  # "n" (nano, ya hecho: colab_1280_s0) o "s" (small)
SEMILLA = 1  # 0 = colab_1280_small_s0 (ya hecho, cumple la regla de 0016); 1 = T3, para confirmarlo (decisión 0016)
NOMBRE = f"colab_1280_{'small' if ESCALA == 's' else 'nano'}_s{SEMILLA}"
LOTE = 8 if ESCALA == "n" else 4  # small pesa más por activación; si CUDA se queda sin memoria, bajar a 2
PROYECTO = RAIZ_DRIVE / "runs"

modelo = YOLO(f"yolo11{ESCALA}-seg.yaml").load(f"yolo11{ESCALA}-seg.pt")  # con C2PSA (arquitectura estándar de Ultralytics)
modelo.train(
    data=str(OUT / "minneapple.yaml"),
    epochs=40,
    imgsz=1280,
    batch=LOTE,
    scale=0.5,
    seed=SEMILLA,
    deterministic=True,
    patience=0,  # sin parada temprana, igual que la base local (decisión 0005)
    cache="disk",
    workers=2,
    mask_ratio=8,
    max_det=300,
    project=str(PROYECTO),
    name=NOMBRE,
    exist_ok=True,
    plots=True,
)
print("Terminado. Resultados en:", PROYECTO / NOMBRE)

## 4. Descargar los resultados
El nombre de la carpeta depende de `ESCALA` (sección 3): `colab_1280_nano_s0` o `colab_1280_small_s0` (el primero, nano, ya se hizo:
`colab_1280_s0`, evaluado y sin mejora suficiente). Copia estos tres archivos desde el panel de archivos de Colab (icono de carpeta,
a la izquierda) o desde Google Drive (`MyDrive/minneapple_colab/runs/<NOMBRE>/`) a tu copia local del repositorio, en
`runs/<NOMBRE>/weights/best.pt`, `runs/<NOMBRE>/results.csv` y `runs/<NOMBRE>/args.yaml` (respeta el nombre `weights`, no `weigths`).

Después, en tu PC (en CPU si la GPU local está entrenando otra cosa):
```
python scripts/evaluar.py --nombre <NOMBRE> --particion val --conf 0.35 --imgsz 1280 --device cpu
```
y compara contra la base 1280 (caja 0,441, máscara 0,345) y, si es la variante `small`, también contra `colab_1280_s0`
(nano-1280: caja 0,443, máscara 0,350) para separar qué aporta el tamaño del modelo de qué aporta la resolución
(decisiones 0016 y 0018).

In [ ]:
from google.colab import files

for archivo in ("weights/best.pt", "results.csv", "args.yaml"):
    ruta = PROYECTO / NOMBRE / archivo
    if ruta.exists():
        print("Descargando", ruta)
        files.download(str(ruta))
    else:
        print("No encontrado (¿terminó el entrenamiento?):", ruta)